# arcroad · Cities open data v01 (standalone)

Collects the **open map counts** for major cities: road length, divided arterial roads, pedestrian crossings, footbridges, traffic signals and turn restrictions from OpenStreetMap, with GHSL population for the same box. It then writes one public table.

- **What becomes public:** only the table from step 4. It holds counts of what the open map records, not measurements and not detour figures.
- **What stays private:** the map extracts and, if you switch it on, the candidate detour points for Indian cities.
- **Weight:** each city downloads a regional map file of 100 MB to 1.5 GB. Allow about 5–15 minutes per city. The 8 major Indian cities take about an hour; the 7 world sample cities a little more.
- **No GitHub needed:** the code is built into this notebook.

Run with Runtime → Run all. Keep the tab open and the computer awake.

In [ ]:
# 1. Setup (about 2 minutes)
!apt-get -qq install -y osmium-tool > /dev/null
!pip -q install osmium rasterio shapely
import base64 as _b64, io as _io, tarfile as _tf, os, sys, json, pathlib, datetime as dt, time, traceback
os.makedirs('/content/arcroad', exist_ok=True)
_BUNDLE = '...'
with _tf.open(fileobj=_io.BytesIO(_b64.b64decode(_BUNDLE)), mode='r:gz') as _t: _t.extractall('/content/arcroad')
sys.path.insert(0, '/content/arcroad')
os.chdir('/content/arcroad')
from tools import multicity_collect as mc, open_data_export as ode
print('code version 52d07cb-standalone | cities known:', len(mc.CITIES), 'India +', len(mc.world_cities()), 'world sample')
from google.colab import drive
drive.mount('/content/drive')
BASE = pathlib.Path('/content/drive/MyDrive/Anthropic/Hyderabad Mobility Atlas/cities')
OUT, PUBLIC = BASE / 'collected', BASE / 'open-data'
for p in (OUT, PUBLIC): p.mkdir(parents=True, exist_ok=True)
CACHE = pathlib.Path('/content/cache'); CACHE.mkdir(exist_ok=True)
print('Results go to Drive:', BASE)


In [ ]:
# 2. Choose cities
# 'india_major'  : Hyderabad, Bengaluru, Chennai, Mumbai, Delhi NCR, Pune, Kolkata, Ahmedabad
# 'india_all'    : the above plus every state and UT capital in the list (about 30)
# 'world_sample' : Manila, Dhaka, Nairobi, New York, London, Bogota, Sao Paulo
# or a list of names, e.g. ['pune', 'world-manila']
CITY_SET = 'india_major'
DETOUR_CANDIDATES = False   # True also stores candidate detour points for Indian cities (private, slow)

SETS = {
    'india_major': ['hyderabad', 'bengaluru', 'chennai', 'mumbai_mmr', 'delhi_ncr', 'pune', 'kolkata', 'ahmedabad'],
    'india_all': sorted(mc.CITIES),
    'world_sample': sorted(mc.world_cities()),
}
CITIES = SETS[CITY_SET] if isinstance(CITY_SET, str) else list(CITY_SET)
unknown = [c for c in CITIES if c not in mc.CITIES and c not in mc.world_cities()]
assert not unknown, f'Unknown city names: {unknown}'
print(len(CITIES), 'cities:', ', '.join(CITIES))


In [ ]:
# 3. Collect, one city at a time. A city already done today is skipped, so re-running resumes.
today = dt.date.today().isoformat()
log = []
for i, city in enumerate(CITIES, 1):
    if (OUT / city / today / 'MANIFEST.json').exists():
        print(f'[{i}/{len(CITIES)}] {city}: already done today'); log.append((city, 'done earlier')); continue
    t0 = time.time()
    print(f'\n[{i}/{len(CITIES)}] {city} ...')
    args = [city, '--out', str(OUT), '--cache', str(CACHE)]
    if DETOUR_CANDIDATES and city in mc.CITIES:
        args.append('--detour')
    try:
        mc.main(args)
        log.append((city, f'ok in {(time.time() - t0) / 60:.1f} min'))
    except SystemExit as e:
        log.append((city, f'stopped: {e}')); print('  stopped:', e)
    except Exception as e:
        log.append((city, f'error: {type(e).__name__}: {e}')); traceback.print_exc()
    # Free disk: regional map files are cached per zone; drop the biggest when space runs low.
    free_gb = __import__('shutil').disk_usage('/content').free / 1e9
    if free_gb < 15:
        for f in sorted(CACHE.glob('*.osm.pbf'), key=lambda p: p.stat().st_size, reverse=True)[:3]:
            f.unlink()
print('\nSummary'); [print(f'  {c}: {s}') for c, s in log]


In [ ]:
# 4. Public table: open-map counts per city (ODbL), for arcroad.org/data
ode.main([str(OUT), str(PUBLIC)])
import pandas as pd
latest = sorted(PUBLIC.glob('arcroad_open_city_counts_*.csv'))[-1]
df = pd.read_csv(latest)
display(df[['city', 'country', 'population_in_box', 'road_km', 'arterial_divided_pct', 'crossings', 'footbridges', 'crossings_per_100k', 'footbridges_per_100k']])
print('Public files in Drive:', latest.name, 'and cities.json')
print('Tell Claude when this is done; it checks the table before anything goes on the site.')
